<h1>Calculating geographic distance

What is the problem with latitude and longitude?

Latitude and longitude are angular coordinates.

Think of the Earth like a globe:

          🌍
       /       \
      /         \
     /           \

Latitude and longitude tell us where we are on that globe.

They aren't like:

5 metres
20 metres
300 metres

where the numbers directly represent physical distance.

So we need a way to convert the coordinates into a system where ordinary geometric distances make sense.

<h3>This is where CRS comes in

You already learned:

CRS = Coordinate Reference System

Its job is basically to tell the computer:

"How should these coordinates be interpreted geographically?"

Our current CRS:

EPSG:4326

means we're using latitude/longitude coordinates based on WGS 84.

That's great for storing and displaying GPS coordinates.


<h3>
But for calculating ordinary distances with GeoPandas, we often want to project those coordinates into a suitable projected CRS where the units are metres.

So:

EPSG:4326
(latitude/longitude)

       ↓

reproject
   
       ↓

Projected CRS
(metres)

In [3]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

In [4]:
data = {
    "site": ["BTS_A", "BTS_B"],
    "latitude": [7.2900, 7.3050],
    "longitude": [5.1200, 5.1450]
}

df = pd.DataFrame(data)

In [5]:
geometry = [
    Point(lon, lat)
    for lon, lat in zip(df["longitude"], df["latitude"])
]

In [6]:
gdf = gpd.GeoDataFrame(
    df,
    geometry=geometry,
    crs="EPSG:4326"
)

<h3>But how the we choose the CRS we're reprojecting to?

We don't need to choose the CRS manually right now.

GeoPandas can suggest and give us an appropriate UTM CRS for the data:

In [8]:
utm_crs = gdf.estimate_utm_crs()

print(utm_crs)

EPSG:32631


<h4>"GeoPandas, look at where these points are and suggest a suitable projected coordinate system."

So now that we know the appropriate CRS, we can create a fresh copy of the gdf which has been reprojected to 32631

In [9]:
gdf_projected = gdf.to_crs(utm_crs)

We can now calculate the distance

In [10]:
#Let's calculate the difference between BTS_A and BTS_B
#BTS_A = row 0
#BTS_B = row 1

distance_m = gdf_projected.geometry[0].distance(
    gdf_projected.geometry[1]
)

#What this does is that it subtracts that of index 1 from 0

In [11]:
print(distance_m)

3221.6002904731886


In [12]:
distance_km = distance_m / 1000

print(distance_km)

3.2216002904731886


The whole process was:
    
Latitude + Longitude

    ↓

GeoDataFrame

        ↓

EPSG:4326

        ↓

Find suitable projected CRS

        ↓

Reproject

        ↓

Coordinates in metres

        ↓

Calculate distance
        
        ↓

Distance in metres
        
        ↓

Convert to km

CRS does NOT calculate the distance.

Instead:

CRS tells us how the coordinates should be interpreted, and an appropriate projected CRS gives us coordinates in a system where geometric distance can be measured meaningfully.

<h1>Distance to the nearest BTS

In [14]:
data_bts = {
    "site": ["BTS_A", "BTS_B", "BTS_C"],
    "latitude": [7.2900, 7.3050, 7.2800],
    "longitude": [5.1200, 5.1450, 5.1000]
}

data_points = {
    "point": ["P1", "P2"],
    "latitude": [7.2950, 7.2750],
    "longitude": [5.1300, 5.1150]
}

In [17]:
#We want to see which base station is closes to which point
#So we'll need two gdfs

df_bts = pd.DataFrame(data_bts)
df_points = pd.DataFrame(data_points)

geometry_bts = [
    Point(lon, lat)
    for lon, lat in zip(df_bts["longitude"], df_bts["latitude"])
]
geometry_points = [
    Point(lon, lat)
    for lon, lat in zip(df_points["longitude"], df_points["latitude"])
]

In [20]:
#Conversion

bts_gdf = gpd.GeoDataFrame(
    df_bts,
    geometry = geometry_bts,
    crs = "EPSG:4326"
)
points_gdf = gpd.GeoDataFrame(
    df_points,
    geometry = geometry_points,
    crs = "EPSG:4326"
)


In [21]:
bts_utm_crs = bts_gdf.estimate_utm_crs()
points_utm_crs = points_gdf.estimate_utm_crs()

print(f"BTS CRS: {bts_utm_crs}. Points CRS: {points_utm_crs}")

BTS CRS: EPSG:32631. Points CRS: EPSG:32631


In [22]:
bts_gdf_projected = bts_gdf.to_crs(bts_utm_crs)
points_gdf_projected = points_gdf.to_crs(points_utm_crs)

In [23]:
print(bts_gdf_projected)
print(points_gdf_projected)

    site  latitude  longitude                       geometry
0  BTS_A     7.290      5.120  POINT (734060.679 806357.219)
1  BTS_B     7.305      5.145  POINT (736814.183 808029.619)
2  BTS_C     7.280      5.100  POINT (731856.734 805240.689)
  point  latitude  longitude                       geometry
0    P1     7.295      5.130  POINT (735162.622 806915.532)
1    P2     7.275      5.115  POINT (733516.172 804695.308)


<h3>When Finding the nearest BTS, how do we do this? Don't worry

GeoPandas has a function specifically for this:

gpd.sjoin_nearest()

It can take every point in your measurement dataset and find the closest feature in the BTS dataset.

In [24]:
nearest = gpd.sjoin_nearest(
    points_gdf_projected,
    bts_gdf_projected,
    how="left",
    distance_col="distance_m"
)

#What this does is that it takes each Points in points_gdf_projected and finds the nearest BTS in bts_gdf_projected, 
#and stores the distance in distance_m column

Let's break that down.

points_gdf_projected

        ↓

For each point
        
        ↓

Find nearest BTS
        
        ↓

Attach BTS information

        ↓

Calculate distance

In [25]:
nearest.head()

,point,latitude_left,longitude_left,geometry,index_right,site,latitude_right,longitude_right,distance_m
0,P1,7.295,5.130,POINT (735162.622 806915.532),0,BTS_A,7.29,5.12,1235.310351
1,P2,7.275,5.115,POINT (733516.172 804695.308),2,BTS_C,7.28,5.10,1746.760798


In [26]:
nearest["distance_km"] = nearest["distance_m"] / 1000

In [27]:
print(
    nearest[
        ["point", "site", "distance_m", "distance_km"]
    ]
)
#Lets' print the necessary info

  point   site   distance_m  distance_km
0    P1  BTS_A  1235.310351     1.235310
1    P2  BTS_C  1746.760798     1.746761


GeoPandas effectively performed:

P1

    ↓

compare against BTS_A, BTS_B, BTS_C
 
    ↓

find minimum distance

    ↓

nearest BTS

P2

    ↓

compare against BTS_A, BTS_B, BTS_C

    ↓

find minimum distance

    ↓

nearest BTS

And it did this without you manually calculating every pair.